## FINAL TRAIN
In previous apporch, I use sequence method, train from General, Legal, Science,... In this Notebook I will train with ONLY 1 dataset with a lot of domain from Multilingual E5 Base 

In [ ]:
DATA_DIR = "/home/nhminh/AI_Project/VietRAG-Embed-E5-Base/database/general_triplet_from_postgres.db"
BASE_MODEL_NAME = "intfloat/multilingual-e5-base"
OUTPUT_DIR = "/kaggle/working/VietRAG"

BATCH_SIZE = 24
EPOCHS = 2

In [ ]:
import math
import sqlite3

from datasets import Features, IterableDataset, Value


def get_data(batch_size=10_000):
    connection = sqlite3.connect(DATA_DIR)
    connection.row_factory = sqlite3.Row
    cursor = connection.execute(
        """
        SELECT anchor, positive, hard_negative
        FROM general_triplet
        ORDER BY data_id
        """
    )

    try:
        while rows := cursor.fetchmany(batch_size):
            for row in rows:
                yield {
                    "anchor": f"query: {row['anchor']}",
                    "positive": f"passage: {row['positive']}",
                    "hard_negative": f"passage: {row['hard_negative']}",
                }
    finally:
        cursor.close()
        connection.close()


with sqlite3.connect(DATA_DIR) as connection:
    num_records = connection.execute(
        "SELECT COUNT(*) FROM general_triplet"
    ).fetchone()[0]

max_steps = math.ceil(num_records / BATCH_SIZE) * EPOCHS

train_dataset = IterableDataset.from_generator(
    get_data,
    features=Features(
        {
            "anchor": Value("string"),
            "positive": Value("string"),
            "hard_negative": Value("string"),
        }
    ),
).shuffle(seed=42, buffer_size=10_000)

In [ ]:
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
    losses,
)

model = SentenceTransformer(BASE_MODEL_NAME)
loss = losses.MultipleNegativesRankingLoss(model)

args = SentenceTransformerTrainingArguments(
    output_dir=OUTPUT_DIR,
    max_steps=max_steps,
    per_device_train_batch_size=BATCH_SIZE,
    learning_rate=5e-6,
    warmup_ratio=0.1,
    fp16=True,
    logging_steps=100,
    save_strategy="steps",
    save_steps=5_000,
    save_total_limit=2,
    report_to="none",
)

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    loss=loss,
)

trainer.train()
trainer.save_model(OUTPUT_DIR)